In [0]:
from pyspark.sql import functions as F
import re


def normalize_table_name(file_name: str) -> str:
    """
    Gera nome de tabela bronze seguro e distinto por extensão.
    Exemplo:
    - downtime_events.csv   -> downtime_events_csv_raw
    - downtime_events.jsonl -> downtime_events_jsonl_raw
    """
    lower_name = file_name.lower()

    if "." in lower_name:
        base_name, extension = lower_name.rsplit(".", 1)
    else:
        base_name, extension = lower_name, "unknown"

    base_name = re.sub(r"[^a-z0-9_]", "_", base_name)
    base_name = re.sub(r"_+", "_", base_name).strip("_")

    extension = re.sub(r"[^a-z0-9_]", "_", extension)
    extension = re.sub(r"_+", "_", extension).strip("_")

    return f"{base_name}_{extension}_raw"


def add_ingestion_metadata(df, file_name: str):
    """
    Adiciona metadados mínimos de ingestão.
    """
    return (
        df.withColumn("ingestion_ts", F.current_timestamp())
          .withColumn("source_file", F.lit(file_name))
    )


def read_file_by_extension(spark, file_path: str):
    """
    Lê arquivo com base na extensão.
    Estratégia:
    - csv  -> lê como CSV
    - json -> tenta JSON permissivo; se vier só _corrupt_record, lê como texto bruto
    - jsonl -> tenta JSON lines; se falhar, tenta CSV
    - xlsx -> lê como Excel
    """
    lower_path = file_path.lower()

    if lower_path.endswith(".csv"):
        return (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file_path)
        )

    elif lower_path.endswith(".json"):
        df_json = (
            spark.read
            .option("multiLine", "false")
            .option("mode", "PERMISSIVE")
            .json(file_path)
        )

        cols = set(df_json.columns)

        # Se só vier _corrupt_record, preserva o bruto como texto
        if cols == {"_corrupt_record"}:
            return spark.read.text(file_path)

        return df_json

    elif lower_path.endswith(".jsonl"):
        try:
            return (
                spark.read
                .option("multiLine", "false")
                .json(file_path)
            )
        except Exception:
            return (
                spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(file_path)
            )

    elif lower_path.endswith(".xlsx"):
        return (
            spark.read
            .format("excel")
            .option("header", "true")
            .load(file_path)
        )

    else:
        raise ValueError(f"Formato não suportado: {file_path}")


def move_to_archive(file_path: str, archive_path: str):
    """
    Move arquivo processado para archive.
    """
    dbutils.fs.mv(file_path, archive_path)